In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import zipfile

with zipfile.ZipFile('archive (1).zip', 'r') as zip_ref:
    zip_ref.extractall('.')

In [6]:
import pandas as pd

df = pd.read_csv('amazon_products.csv')
cats = pd.read_csv('amazon_categories.csv')

df.head()

,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black...",https://m.media-amazon.com/images/I/815dLQKYIY...,https://www.amazon.com/dp/B014TMV5YE,4.5,0,139.99,0.00,104,False,2000
1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcas...,https://m.media-amazon.com/images/I/81bQlm7vf6...,https://www.amazon.com/dp/B07GDLCQXV,4.5,0,169.99,209.99,104,False,1000
2,B07XSCCZYG,Platinum Elite Softside Expandable Checked Lug...,https://m.media-amazon.com/images/I/71EA35zvJB...,https://www.amazon.com/dp/B07XSCCZYG,4.6,0,365.49,429.99,104,False,300
3,B08MVFKGJM,Freeform Hardside Expandable with Double Spinn...,https://m.media-amazon.com/images/I/91k6NYLQyI...,https://www.amazon.com/dp/B08MVFKGJM,4.6,0,291.59,354.37,104,False,400
4,B01DJLKZBA,Winfield 2 Hardside Expandable Luggage with Sp...,https://m.media-amazon.com/images/I/61NJoaZcP9...,https://www.amazon.com/dp/B01DJLKZBA,4.5,0,174.99,309.99,104,False,400


In [7]:
df.shape

(1426337, 11)

In [8]:
df.columns

Index(['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price',
       'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth'],
      dtype='object')

In [9]:
df.isnull().sum()

,0
asin,0
title,1
imgUrl,0
productURL,0
stars,0
reviews,0
price,0
listPrice,0
category_id,0
isBestSeller,0


In [10]:
df = df.dropna(subset = ['title'])
df.shape

(1426336, 11)

In [11]:
df['boughtInLastMonth'].describe()

,boughtInLastMonth
count,1.426336e+06
mean,1.419824e+02
std,8.362722e+02
min,0.000000e+00
25%,0.000000e+00
50%,0.000000e+00
75%,5.000000e+01
max,1.000000e+05


In [12]:
df['success'] = (df['boughtInLastMonth'] > 50).astype(int)
df['success'].value_counts()

,count
success,
0,1074871
1,351465


In [13]:
df = df.merge(cats, left_on='category_id', right_on='id', how='left')
df[['title', 'category_name', 'success']].head()

,title,category_name,success
0,"Sion Softside Expandable Roller Luggage, Black...",Suitcases,1
1,Luggage Sets Expandable PC+ABS Durable Suitcas...,Suitcases,1
2,Platinum Elite Softside Expandable Checked Lug...,Suitcases,1
3,Freeform Hardside Expandable with Double Spinn...,Suitcases,1
4,Winfield 2 Hardside Expandable Luggage with Sp...,Suitcases,1


In [14]:
df.shape

(1426336, 14)

In [15]:
df['category_name'].nunique()

248

In [16]:
df['category_name'].value_counts()

,count
category_name,
Girls' Clothing,28619
Boys' Clothing,24660
Toys & Games,20846
Men's Shoes,19822
Women's Handbags,18994
...,...
Smart Home: Lawn and Garden,76
Smart Home: Voice Assistants and Hubs,50
Smart Home: New Smart Devices,42


In [17]:
category_counts = df['category_name'].value_counts()
top_categories = category_counts[category_counts >= 7000].index
df['category_grouped'] = df['category_name'].where(df['category_name'].isin(top_categories), 'Other')

print(df['category_grouped'].nunique())
print(df['category_grouped'].value_counts()['Other'] / len(df))

95
0.3422748917506114


In [18]:
category_dummies = pd.get_dummies(df['category_grouped'], prefix='cat')
category_dummies.shape

(1426336, 95)

In [19]:
import numpy as np

df['discount_pct'] = (df['listPrice'] - df['price']) / df['listPrice']
df['discount_pct'] = df['discount_pct'].replace([np.inf, -np.inf], 0)
df['discount_pct'] = df['discount_pct'].fillna(0)
df[['price', 'listPrice', 'discount_pct']].head()

,price,listPrice,discount_pct
0,139.99,0.00,0.000000
1,169.99,209.99,0.190485
2,365.49,429.99,0.150003
3,291.59,354.37,0.177159
4,174.99,309.99,0.435498


In [20]:
df[['price', 'stars', 'reviews', 'discount_pct']].isnull().sum()

,0
price,0
stars,0
reviews,0
discount_pct,0


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300, stop_words='english')
title_features = tfidf.fit_transform(df['title'])
title_features.shape

(1426336, 300)

In [22]:
df_sample = df.sample(n=5000, random_state=42).reset_index(drop=True)
df_sample.shape

(5000, 16)

In [23]:
import requests
from PIL import Image
from io import BytesIO
import os

os.makedirs('images', exist_ok=True)

def download_image(url, save_path):
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert('RGB')
        img.save(save_path)
        return True
    except Exception as e:
        return False

success_count = 0
for i, row in df_sample.iterrows():
    save_path = f"images/{i}.jpg"
    if download_image(row['imgUrl'], save_path):
        success_count += 1
    if i % 500 == 0:
        print(f"{i} ürün işlendi, {success_count} başarılı")

print(f"Toplam: {success_count} / {len(df_sample)} fotoğraf indirildi")

0 ürün işlendi, 1 başarılı
500 ürün işlendi, 501 başarılı
1000 ürün işlendi, 1001 başarılı
1500 ürün işlendi, 1501 başarılı
2000 ürün işlendi, 2001 başarılı
2500 ürün işlendi, 2501 başarılı
3000 ürün işlendi, 3001 başarılı
3500 ürün işlendi, 3500 başarılı
4000 ürün işlendi, 4000 başarılı
4500 ürün işlendi, 4498 başarılı
Toplam: 4997 / 5000 fotoğraf indirildi


In [24]:
!pip install tensorflow -q

In [25]:
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np

resnet_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [26]:
def get_image_features(img_path):
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)
        features = resnet_model.predict(img_array, verbose=0)
        return features.flatten()
    except Exception as e:
        return None

image_features_list = []
valid_indices = []

for i in range(len(df_sample)):
    img_path = f"images/{i}.jpg"
    if os.path.exists(img_path):
        features = get_image_features(img_path)
        if features is not None:
            image_features_list.append(features)
            valid_indices.append(i)
    if i % 500 == 0:
        print(f"{i} fotoğraf işlendi")

print(f"Toplam işlenen: {len(image_features_list)} fotoğraf")

0 fotoğraf işlendi
500 fotoğraf işlendi
1000 fotoğraf işlendi
1500 fotoğraf işlendi
2000 fotoğraf işlendi
2500 fotoğraf işlendi
3000 fotoğraf işlendi
3500 fotoğraf işlendi
4000 fotoğraf işlendi
4500 fotoğraf işlendi
Toplam işlenen: 4997 fotoğraf


In [27]:
image_features_array = np.array(image_features_list)
print("Görsel özellik boyutu:", image_features_array.shape)

df_valid = df_sample.iloc[valid_indices].reset_index(drop=True)
df_valid.shape

Görsel özellik boyutu: (4997, 2048)


(4997, 16)

In [28]:
from scipy.sparse import hstack, csr_matrix

In [29]:
category_dummies_v2 = pd.get_dummies(df_valid['category_grouped'], prefix='cat')
title_features_v2 = tfidf.transform(df_valid['title'])
numeric_features_v2 = df_valid[['price', 'stars', 'reviews', 'discount_pct']].values

X_with_images = hstack([
    csr_matrix(numeric_features_v2),
    csr_matrix(category_dummies_v2.values),
    title_features_v2,
    csr_matrix(image_features_array)
])

y_with_images = df_valid['success']

X_with_images.shape

(4997, 2447)

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [31]:
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_with_images, y_with_images, test_size=0.2, random_state=42, stratify=y_with_images
)

scale_pos_weight_img = (y_train_img == 0).sum() / (y_train_img == 1).sum()

model_with_images = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight_img
)
model_with_images.fit(X_train_img, y_train_img)

y_pred_img = model_with_images.predict(X_test_img)

print("Accuracy:", accuracy_score(y_test_img, y_pred_img))
print(classification_report(y_test_img, y_pred_img))

Accuracy: 0.78
              precision    recall  f1-score   support

           0       0.80      0.94      0.87       757
           1       0.60      0.28      0.38       243

    accuracy                           0.78      1000
   macro avg       0.70      0.61      0.62      1000
weighted avg       0.75      0.78      0.75      1000



In [32]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
image_features_reduced = pca.fit_transform(image_features_array)

print("Önceki boyut:", image_features_array.shape)
print("Yeni boyut:", image_features_reduced.shape)
print("Korunan bilgi oranı:", pca.explained_variance_ratio_.sum())

Önceki boyut: (4997, 2048)
Yeni boyut: (4997, 50)
Korunan bilgi oranı: 0.55525166


In [33]:
X_with_images_v2 = hstack([
    csr_matrix(numeric_features_v2),
    csr_matrix(category_dummies_v2.values),
    title_features_v2,
    csr_matrix(image_features_reduced)
])

y_with_images_v2 = df_valid['success']

X_train_img2, X_test_img2, y_train_img2, y_test_img2 = train_test_split(
    X_with_images_v2, y_with_images_v2, test_size=0.2, random_state=42, stratify=y_with_images_v2
)

scale_pos_weight_img2 = (y_train_img2 == 0).sum() / (y_train_img2 == 1).sum()

model_with_images_v2 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight_img2
)
model_with_images_v2.fit(X_train_img2, y_train_img2)

y_pred_img2 = model_with_images_v2.predict(X_test_img2)

print("Accuracy:", accuracy_score(y_test_img2, y_pred_img2))
print(classification_report(y_test_img2, y_pred_img2))

Accuracy: 0.788
              precision    recall  f1-score   support

           0       0.84      0.89      0.86       757
           1       0.58      0.48      0.52       243

    accuracy                           0.79      1000
   macro avg       0.71      0.68      0.69      1000
weighted avg       0.78      0.79      0.78      1000



In [34]:
df_sample2 = df.sample(n=20000, random_state=42).reset_index(drop=True)
df_sample2.shape

(20000, 16)

In [35]:
os.makedirs('images2', exist_ok=True)

success_count2 = 0
for i, row in df_sample2.iterrows():
    save_path = f"images2/{i}.jpg"
    if download_image(row['imgUrl'], save_path):
        success_count2 += 1
    if i % 2000 == 0:
        print(f"{i} ürün işlendi, {success_count2} başarılı")

print(f"Toplam: {success_count2} / {len(df_sample2)} fotoğraf indirildi")

0 ürün işlendi, 1 başarılı
2000 ürün işlendi, 2000 başarılı
4000 ürün işlendi, 4000 başarılı
6000 ürün işlendi, 5999 başarılı
8000 ürün işlendi, 7999 başarılı
10000 ürün işlendi, 9997 başarılı
12000 ürün işlendi, 11995 başarılı
14000 ürün işlendi, 13993 başarılı
16000 ürün işlendi, 15992 başarılı
18000 ürün işlendi, 17990 başarılı
Toplam: 19986 / 20000 fotoğraf indirildi


In [36]:
image_features_list2 = []
valid_indices2 = []

def get_image_features2(img_path):
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)
        features = resnet_model.predict(img_array, verbose=0)
        return features.flatten()
    except Exception as e:
        return None

for i in range(len(df_sample2)):
    img_path = f"images2/{i}.jpg"
    if os.path.exists(img_path):
        features = get_image_features2(img_path)
        if features is not None:
            image_features_list2.append(features)
            valid_indices2.append(i)
    if i % 2000 == 0:
        print(f"{i} fotoğraf işlendi")

print(f"Toplam işlenen: {len(image_features_list2)} fotoğraf")

0 fotoğraf işlendi
2000 fotoğraf işlendi
4000 fotoğraf işlendi
6000 fotoğraf işlendi
8000 fotoğraf işlendi
10000 fotoğraf işlendi
12000 fotoğraf işlendi
14000 fotoğraf işlendi
16000 fotoğraf işlendi
18000 fotoğraf işlendi
Toplam işlenen: 19986 fotoğraf


In [37]:
import numpy as np

image_features_array2 = np.array(image_features_list2)
print("Boyut:", image_features_array2.shape)

np.save('/content/drive/MyDrive/image_features_array2.npy', image_features_array2)
np.save('/content/drive/MyDrive/valid_indices2.npy', np.array(valid_indices2))

print("Drive'a kaydedildi!")

Boyut: (19986, 2048)
Drive'a kaydedildi!


In [38]:
df_sample2.to_pickle('/content/drive/MyDrive/df_sample2.pkl')
print("df_sample2 de kaydedildi!")

df_sample2 de kaydedildi!


In [39]:
from sklearn.decomposition import PCA

pca2 = PCA(n_components=50, random_state=42)
image_features_reduced2 = pca2.fit_transform(image_features_array2)

print("Önceki boyut:", image_features_array2.shape)
print("Yeni boyut:", image_features_reduced2.shape)
print("Korunan bilgi oranı:", pca2.explained_variance_ratio_.sum())

Önceki boyut: (19986, 2048)
Yeni boyut: (19986, 50)
Korunan bilgi oranı: 0.5501407


In [40]:
df_valid2 = df_sample2.iloc[valid_indices2].reset_index(drop=True)

category_dummies_v3 = pd.get_dummies(df_valid2['category_grouped'], prefix='cat')
title_features_v3 = tfidf.transform(df_valid2['title'])
numeric_features_v3 = df_valid2[['price', 'stars', 'reviews', 'discount_pct']].values

df_valid2.shape

(19986, 16)

In [41]:
X_with_images_v3 = hstack([
    csr_matrix(numeric_features_v3),
    csr_matrix(category_dummies_v3.values),
    title_features_v3,
    csr_matrix(image_features_reduced2)
])

y_with_images_v3 = df_valid2['success']

X_train_img3, X_test_img3, y_train_img3, y_test_img3 = train_test_split(
    X_with_images_v3, y_with_images_v3, test_size=0.2, random_state=42, stratify=y_with_images_v3
)

scale_pos_weight_img3 = (y_train_img3 == 0).sum() / (y_train_img3 == 1).sum()

model_with_images_v3 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight_img3
)
model_with_images_v3.fit(X_train_img3, y_train_img3)

y_pred_img3 = model_with_images_v3.predict(X_test_img3)

print("Accuracy:", accuracy_score(y_test_img3, y_pred_img3))
print(classification_report(y_test_img3, y_pred_img3))

Accuracy: 0.7856428214107054
              precision    recall  f1-score   support

           0       0.88      0.84      0.85      3020
           1       0.55      0.63      0.59       978

    accuracy                           0.79      3998
   macro avg       0.71      0.73      0.72      3998
weighted avg       0.80      0.79      0.79      3998



In [42]:
import joblib

joblib.dump(model_with_images_v3, '/content/drive/MyDrive/model_with_images_v3.pkl')
joblib.dump(pca2, '/content/drive/MyDrive/pca2.pkl')

print("Modeller Drive'a kaydedildi!")

Modeller Drive'a kaydedildi!


In [43]:
# Test için, veri setinden rastgele bir ürün fotoğrafı seçelim
test_product = df_valid2.iloc[0]
test_img_url = test_product['imgUrl']

print("Test ürünü:", test_product['title'])
print("Fotoğraf linki:", test_img_url)

# 1) Fotoğrafı indir ve işle
download_image(test_img_url, 'test_product.jpg')
test_image_features = get_image_features2('test_product.jpg')
test_image_reduced = pca2.transform(test_image_features.reshape(1, -1))

# 2) Diğer özellikleri hazırla (aynı ürünün gerçek bilgileriyle)
test_title_features = tfidf.transform([test_product['title']])

test_category_dummies = pd.DataFrame(0, index=[0], columns=category_dummies_v3.columns)
cat_col = 'cat_' + test_product['category_grouped']
if cat_col in test_category_dummies.columns:
    test_category_dummies[cat_col] = 1
else:
    test_category_dummies['cat_Other'] = 1

test_numeric = np.array([[test_product['price'], test_product['stars'],
                           test_product['reviews'], test_product['discount_pct']]])

# 3) Hepsini birleştir
X_test_product = hstack([
    csr_matrix(test_numeric),
    csr_matrix(test_category_dummies.values),
    test_title_features,
    csr_matrix(test_image_reduced)
])

# 4) Tahmin al
pred = model_with_images_v3.predict(X_test_product)
prob = model_with_images_v3.predict_proba(X_test_product)

print("\nTahmin (0=başarısız, 1=başarılı):", pred[0])
print("Başarı olasılığı: %{:.1f}".format(prob[0][1] * 100))
print("Gerçek durum (success):", test_product['success'])

Test ürünü: LOVEVOOK Diaper Bag Backpack, Diaper Backpack with Portable Changing Pad & Pacifier Case,Multipurpose Travel Back Pack for Moms Dads
Fotoğraf linki: https://m.media-amazon.com/images/I/718pt0oLESL._AC_UL320_.jpg

Tahmin (0=başarısız, 1=başarılı): 0
Başarı olasılığı: %23.9
Gerçek durum (success): 0


In [44]:
# success = 1 olan ürünlerden birini seçelim
successful_products = df_valid2[df_valid2['success'] == 1]
test_product2 = successful_products.iloc[0]
test_img_url2 = test_product2['imgUrl']

print("Test ürünü:", test_product2['title'])
print("Fotoğraf linki:", test_img_url2)

# 1) Fotoğrafı indir ve işle
download_image(test_img_url2, 'test_product2.jpg')
test_image_features2 = get_image_features2('test_product2.jpg')
test_image_reduced2 = pca2.transform(test_image_features2.reshape(1, -1))

# 2) Diğer özellikleri hazırla
test_title_features2 = tfidf.transform([test_product2['title']])

test_category_dummies2 = pd.DataFrame(0, index=[0], columns=category_dummies_v3.columns)
cat_col2 = 'cat_' + test_product2['category_grouped']
if cat_col2 in test_category_dummies2.columns:
    test_category_dummies2[cat_col2] = 1
else:
    test_category_dummies2['cat_Other'] = 1

test_numeric2 = np.array([[test_product2['price'], test_product2['stars'],
                            test_product2['reviews'], test_product2['discount_pct']]])

# 3) Hepsini birleştir
X_test_product2 = hstack([
    csr_matrix(test_numeric2),
    csr_matrix(test_category_dummies2.values),
    test_title_features2,
    csr_matrix(test_image_reduced2)
])

# 4) Tahmin al
pred2 = model_with_images_v3.predict(X_test_product2)
prob2 = model_with_images_v3.predict_proba(X_test_product2)

print("\nTahmin (0=başarısız, 1=başarılı):", pred2[0])
print("Başarı olasılığı: %{:.1f}".format(prob2[0][1] * 100))
print("Gerçek durum (success):", test_product2['success'])

Test ürünü: Spry Xylitol Toothpaste 5oz, Fluoride Toothpaste Adult and Kids, Teeth Whitening Toothpaste with Xylitol, Natural Breath Freshening, Mouth Moisturizing Ingredients, Spearmint (Pack of 1)
Fotoğraf linki: https://m.media-amazon.com/images/I/51I687ieheS._AC_UL320_.jpg

Tahmin (0=başarısız, 1=başarılı): 1
Başarı olasılığı: %53.6
Gerçek durum (success): 1
